In [73]:
from src.existingSimulator.simulator import simulateExperiment,exponential_lambda_search,calc_lambda

alpha,beta,beta_hat,time_spent,convergence= simulateExperiment(calc_lambda)

print("Single execution with binary+exponential lambda search:", alpha.item(), beta.item(), beta_hat.item(), time_spent, convergence)

Using binary search for lambda.
Single execution with binary+exponential lambda search: 0.4000083804130554 0.08481208980083466 0.3333333432674408 [0.04781293869018555] [99854.328125, 76450.90625, 34096.0390625, 8034.453125, 15570.65625, 4487.59375, 1581.5390625, 1499.421875, 29.296875, 737.9765625, 355.0859375, 163.0859375, 66.9609375, 18.8359375, 5.2109375, 6.8203125, 0.8046875]


In [74]:
from time import time

import torch
import math
# NewtonImplentation
@torch.compile
def calc_derivative(rho, beta):
    fraction_term = torch.sum(rho * beta, dim=1, keepdim=True)
    complete_middle_block = -rho + fraction_term
    fp = -torch.sum(
        beta * complete_middle_block * (torch.log2(beta + 1e-10) + 1/math.log(2)),
        dim=(1, 2, 3)
    )
    return fp


def calc_newton_lambda(rho, m, objective, xtol=1e-3, max_iter=20):
    print("Calculating lambda using Newton's method...")
    device = rho.device
    B = rho.shape[0]
    if isinstance(m, (int, float)):
        m = torch.full((B,), m, dtype=rho.dtype, device=device)
    lbdr, lbdh = exponential_lambda_search(rho, m, objective)
    lbd = (lbdh +lbdr) / 2

    last_lbd = lbd
    convergence = []
    for i in range(max_iter):
        beta, h_hat = objective(rho, lbd.view(-1, 1, 1, 1))
        f = h_hat - m
        f_derivated = calc_derivative(rho, beta)
        convergence.append(f.abs().max().item())
        
        if f.abs().max() < xtol:
            break

        lbd = lbd - f / f_derivated
        lbd = torch.clamp(lbd, lbdr, lbdh)

        if(lbd - last_lbd).abs().max() < xtol:
            break
        last_lbd = lbd

    _, h_hat = objective(rho, lbd.view(-1, 1, 1, 1))
    return lbd, h_hat, convergence

test_Success = True
for q_val in [2, 3, 6, 9]:
    res = simulateExperiment(calc_newton_lambda, alpha=0.1, q=q_val)
    res2 = simulateExperiment(calc_lambda, alpha=0.1, q=q_val)
    
    alpha_ok = (res[0] - res2[0]).abs().item() < 1e-3
    beta_ok = (res[1] - res2[1]).abs().item() < 1e-3
    beta_hat_ok = (res[2] - res2[2]).abs().item() < 1e-3
    
    ok = alpha_ok and beta_ok and beta_hat_ok
    test_Success = test_Success and ok

print("\n\nTest results for q values 2, 3, 6, 9:")    
if test_Success:
    print("Test passed: Both methods have the same results.")
else:
    print("Test failed: Methods have different results.")

Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.


Test results for q values 2, 3, 6, 9:
Test passed: Both methods have the same results.


In [76]:
import pandas as pd
import numpy as np

alphas = [0.1,0.2,0.3,0.4,0.5]
q = [2,3,6,9]
convergence_rows = []
time_rows = []
repetitions = 10

for q_val in q:
    for alpha in alphas:
        for repetition in range(repetitions):
            for function, name in [(calc_newton_lambda, "Newton's Method"), (calc_lambda, "Binary Search")]:
                alpha_hat, beta, beta_hat, time_spent, convergence = simulateExperiment(function, alpha, q_val)
                for i, error in enumerate(convergence):
                    convergence_rows.append({"repetition": repetition, "iteration": i, "algorithm": name, "error": error, "alpha": alpha, "q": q_val})
                time_rows.append({"repetition": repetition, "algorithm": name, "time": np.mean(time_spent), "q": q_val, "alpha": alpha})

convergenceDf = pd.DataFrame(convergence_rows, columns=["repetition", "iteration", "algorithm", "error","alpha", "q"]).groupby(["iteration", "algorithm", "alpha", "q"]).mean().reset_index()
timeDF = pd.DataFrame(time_rows, columns=["repetition", "algorithm", "time", "q", "alpha"]).groupby(["algorithm", "q", "alpha"]).mean().reset_index()

Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating lambda using Newton's method...
Using binary search for lambda.
Calculating 

In [79]:
import plotly.express as px
 
fig = px.line(convergenceDf, x="iteration", y="error",log_y=True, color="alpha",facet_col="algorithm", facet_row="q",title="Convergence of Newton's Method vs Binary Search for alpha values: 0.1 to 0.5")
fig.show()
fig.write_html("convergence_data.html")
timeDF["q"] = timeDF["q"].astype(str)
timeDF["alpha"] = timeDF["alpha"].astype(str)

newton_times = timeDF[timeDF["algorithm"] == "Newton's Method"].groupby(["q","alpha"])["time"].mean()
binary_times = timeDF[timeDF["algorithm"] == "Binary Search"].groupby(["q","alpha"])["time"].mean()
diff = (newton_times - binary_times) / binary_times * 100
fig2 = px.bar(diff.reset_index(), x="q", y="time", color="alpha", barmode="group",title="Percentage Difference in Time Spent (Newton's Method vs Binary Search)")
fig2.update_layout(yaxis_title="Percentage Difference (%)")
fig2.show()
fig2.write_html("time_comparison.html") 